# BANKING77 Part 3: Structured JSON Output, Validation and Mock Tools
**Run:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all.

Flow tested here: `Customer request -> Qwen JSON action -> schema validation -> mock tool -> final response`.
All tools use mock data only; no real banking data or credentials.

## 1 - Get the project code and install dependencies

In [ ]:
import os
import shutil

# 1. Reset back to the absolute base directory
%cd /content

# 2. Clean up any nested or corrupted project folders
if os.path.isdir("project"):
    print("Cleaning up existing project directory...")
    shutil.rmtree("project")

# 3. Clone fresh to /content/project
BRANCH = "2025paml541-Karthik"
REPO = "https://github.com/2025paml620/AppliedGenAI-Banking77-Project-Group2.git"

!git clone -q --branch {BRANCH} {REPO} project

# 4. Change directory to the clean project root
%cd /content/project

# 5. Re-install requirements in the correct location
%pip -q install "transformers==4.57.6" "datasets==3.6.0" "accelerate>=0.33" pandas pytest

## 2 - Unit tests (no model needed): valid and invalid JSON, tool failures, guardrails

In [ ]:
!python -m pytest tests/test_structured.py -v

## 3 - Prompt template and schema validation

In [ ]:
import json, pandas as pd
from run_part3 import load_intents, run_demo, DEMO_QUERIES
from src.prompts.structured import build_structured_prompt
from src.structured.schema import validate_output

intents = load_intents("data/evaluation_sample.csv")
print(len(intents), "labels")
print(build_structured_prompt("Where is TXN1002?", intents)[-900:])

In [ ]:
good = '{"intent": "lost_or_stolen_card", "response": "Please use the official app.", "action_required": false, "action": null}'
samples = {
    "valid": good,
    "markdown fence": "```json\n" + good + "\n```",
    "not json": "Sure! Your card is blocked.",
    "bad label": good.replace("lost_or_stolen_card", "made_up"),
    "missing key": '{"intent": "lost_or_stolen_card"}',
    "action mismatch": good.replace('"action_required": false', '"action_required": true'),
}
for name, raw in samples.items():
    data, errors = validate_output(raw, intents)
    print(f"{name:16} -> {'OK' if data else errors}")

## 4 - Load Qwen2.5-1.5B-Instruct

In [ ]:
from src.model.llm_baseline import load_model
model, tokenizer = load_model()

## 5 - End-to-end demo: tool calls, missing/unknown arguments, blocked input

In [ ]:
pd.set_option("display.max_colwidth", 90, "display.width", 250)
df = run_demo(model, tokenizer, intents)
display(df[["query", "status", "intent", "tool", "final_response"]])
print(df["status"].value_counts().to_dict())

Raw model output and validation errors (for failure analysis):

In [ ]:
for _, r in df.iterrows():
    print("Q:", r["query"]); print("  raw:", r["raw_output"]); print("  errors:", r["errors"] or "-"); print()

## 6 - RAG: retrieve similar labelled train examples to improve intent accuracy
Corpus = BANKING77 **train** split; evaluation rows come from the **test** split (no leakage).

In [ ]:
from src.rag.retriever import build_retriever
retriever = build_retriever()
for hit in retriever.retrieve("I lost my card while travelling", k=5):
    print(round(hit["score"], 3), hit["intent"], "|", hit["text"])

In [ ]:
# Same test examples, with and without RAG (about 77 queries; takes a few minutes on a T4)
!python -m scripts.evaluate_rag --limit 77 --k 5

In [ ]:
print(open("experiments/rag_metrics.json").read())
cmp = pd.read_csv("experiments/rag_comparison.csv")
display(cmp[cmp["no_rag_pred"] != cmp["rag_pred"]].head(15))   # where RAG changed the answer

## 7 - Try your own query

In [ ]:
from src.model.llm_baseline import generate_response
from src.structured.pipeline import handle_query
gen = lambda p: generate_response(model, tokenizer, p, {"do_sample": False, "max_new_tokens": 200})
result = handle_query("Block my card ending 5678", intents, gen, retriever=retriever, k=5)
print(json.dumps(result, indent=2, default=str))

## 8 - Notes
- Failures never reach the user as raw model text: invalid JSON, unsafe text and tool errors all return fixed/explanatory messages.
- Tool-action replies are built from the tool result, not from model text.
- Small models may break the JSON format or omit an action; the status column shows how often (`invalid_output`).
- Adaptation comparison (prompting vs RAG vs PEFT): see `reports/PART3_REPORT.md`.